In [0]:
%run ./env

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./delta_schema

In [0]:
%run ./load_data

In [0]:
%run ./transform_data

In [0]:
# Controle en debut de job : l'equipe energie peut modifier sa configuration sans
# prevenir. Si un changement est detecte, le job echoue AVANT tout chargement : le
# rapport reste sur les donnees de la veille plutot que d'afficher une donnee douteuse,
# et la notification d'echec Databricks previent.
# Seule exception voulue : un simple renommage de code (meme id) ne bloque pas, le
# rapport continue de fonctionner par l'id.
ligne_controlee = "rouen1"
table_mapping = current_catalog + "." + current_schema + ".dim_report_parameter_mapping"
table_photo_config = current_catalog + "." + current_schema + ".controle_config_source"

# Saisies manuelles, hors energy_metadata (meme liste que codes_manuels_attendus
# dans fact_batch_measures, qui n'est pas encore execute a ce stade)
codes_manuels_rapport = ["malt_weight", "goods_weight", "malt_yield_r2"]

parametres_actifs = parameters_variables.filter(~F.coalesce(F.col("deleted"), F.lit(False)))

nb_anomalies_codes = 0
nb_anomalies_metadata = 0
if spark.catalog.tableExists(table_mapping):
    mapping_cible = spark.table(table_mapping)

    # Parametre du rapport supprime ou introuvable dans la source
    anomalies_codes = (
        mapping_cible.alias("m")
        .join(
            parameters_variables.alias("pv"),
            F.col("m.parameter_id") == F.col("pv.id_parameter_variable"),
            "left"
        )
        .filter(
            F.col("pv.id_parameter_variable").isNull()
            | F.coalesce(F.col("pv.deleted"), F.lit(False))
        )
        .select(
            F.col("m.display_order"),
            F.col("m.column_label"),
            F.col("m.code").alias("code_mapping"),
            F.col("pv.code").alias("code_source"),
            F.col("pv.deleted").alias("supprime_source")
        )
    )

    # Parametre automatique dont le code actuel dans la source est absent
    # d'energy_metadata : ses mesures ne seraient plus chargees. On part du code de
    # la source (par l'id) et non de celui du mapping, pour qu'un renommage coherent
    # des deux cotes ne declenche rien.
    anomalies_metadata = (
        mapping_cible.alias("m")
        .join(
            parameters_variables.alias("pv"),
            F.col("m.parameter_id") == F.col("pv.id_parameter_variable"),
            "inner"
        )
        .filter(
            ~F.col("m.code").isin(codes_manuels_rapport)
            & ~F.col("pv.code").isin(codes_manuels_rapport)
        )
        .join(
            energy_metadata.select(F.col("measure_name").alias("code_metadata")).distinct(),
            F.col("pv.code") == F.col("code_metadata"),
            "left_anti"
        )
        .select(
            F.col("m.display_order"),
            F.col("m.column_label"),
            F.col("m.code").alias("code_mapping"),
            F.col("pv.code").alias("code_source")
        )
    )

    nb_anomalies_codes = anomalies_codes.count()
    nb_anomalies_metadata = anomalies_metadata.count()

    if nb_anomalies_codes > 0:
        print(f"ATTENTION : {nb_anomalies_codes} indicateur(s) du rapport supprime(s) ou introuvable(s) dans la source :")
        anomalies_codes.orderBy("display_order").show(truncate=False)

    if nb_anomalies_metadata > 0:
        print(f"ATTENTION : {nb_anomalies_metadata} indicateur(s) du rapport absents d'energy_metadata, leurs mesures ne seraient plus chargees :")
        anomalies_metadata.orderBy("display_order").show(truncate=False)

# Photo des mesures energie declarees pour la ligne dans energy_metadata, reperees
# par l'id de leur parametre dans la source. Toute difference avec la photo validee
# bloque le job : mesure ajoutee ou retiree, bascule vers un autre id, renommage
# fait d'un seul cote... Seule la liste des mesures compte (pas l'unite, la regle de
# calcul ou le libelle), et un renommage fait des deux cotes garde le meme id.
# La photo n'est remplacee que si rien n'a change : le job reste bloque tant que le
# changement n'a pas ete accepte (DROP de la table photo, cf. message d'erreur).
config_actuelle = (
    energy_metadata.filter(F.col("prd_line") == ligne_controlee).alias("em")
    .join(
        parametres_actifs.alias("pv"),
        F.col("em.measure_name") == F.col("pv.code"),
        "left"
    )
    .select(
        # Mesure sans parametre correspondant dans la source : reperee par son code
        F.coalesce(
            F.col("pv.id_parameter_variable").cast("string"),
            F.concat(F.lit("code:"), F.col("em.measure_name"))
        ).alias("cle"),
        F.col("em.measure_name").alias("code")
    )
    .distinct()
)

nb_changements_config = 0
if spark.catalog.tableExists(table_photo_config):
    photo_validee = spark.table(table_photo_config)
    mesures_retirees = photo_validee.join(config_actuelle, ["cle"], "left_anti")
    mesures_ajoutees = config_actuelle.join(photo_validee, ["cle"], "left_anti")
    nb_changements_config = mesures_retirees.count() + mesures_ajoutees.count()

    if nb_changements_config > 0:
        print(f"ATTENTION : les mesures energie de {ligne_controlee} ont change cote source.")
        print("Mesures retirees (id ou code) :")
        mesures_retirees.orderBy("cle").show(truncate=False)
        print("Mesures ajoutees (id ou code) :")
        mesures_ajoutees.orderBy("cle").show(truncate=False)
else:
    print(f"{table_photo_config} : premiere photo de la configuration, aucune comparaison ce run")

if nb_anomalies_codes + nb_anomalies_metadata + nb_changements_config > 0:
    raise Exception(
        "Parametres energie modifies cote source : aucune donnee chargee ce run "
        "(detail dans la sortie ci-dessus). Indicateur supprime ou absent "
        "d'energy_metadata : corriger dim_report_parameter_mapping. Mesures "
        f"ajoutees ou retirees : une fois traite, executer DROP TABLE {table_photo_config} "
        "puis relancer le job."
    )

# Aucun changement : la photo est rafraichie (codes renommes a jour) et le job continue
config_actuelle.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_photo_config)

In [0]:
%run ./dim_batch

In [0]:
%run ./dim_parameter

In [0]:
%run ./dim_report_parameter_mapping

In [0]:
%run ./fact_batch_measures